In [ ]:
!pip install gurobipy
from gurobipy import Model, GRB, quicksum
import pandas as pd



plants = ["P1","P2","P3"]
plant_cap = {"P1":65, "P2":50, "P3":70}

dcs = ["D1","D2","D3","D4","D5","D6"]
dc_cost = {"D1":4500,"D2":5000,"D3":6000,"D4":4300,"D5":4400,"D6":3300}
dc_cap = {"D1":50,"D2":49,"D3":40,"D4":41,"D5":42,"D6":43}
dc_handle = {"D1":1.8,"D2":1.5,"D3":2.3,"D4":2.5,"D5":1.3,"D6":2.0}
dc_open = {"D1":1,"D2":0,"D3":0,"D4":1,"D5":1,"D6":1}

retails = ["R1","R2","R3","R4","R5","R6","R7","R8","R9","R10","R11"]
ret_demand = {"R1":20,"R2":15,"R3":13,"R4":12,"R5":13,"R6":18,
              "R7":13,"R8":7,"R9":13,"R10":9,"R11":17}

dist_pd = {
 "P1":{"D1":11,"D2":13,"D3":14,"D4":16,"D5":11,"D6":11},
 "P2":{"D1":12,"D2":12,"D3":12,"D4":14,"D5":13,"D6":14},
 "P3":{"D1":14,"D2":11,"D3":17,"D4":12,"D5":10,"D6":12}
}

dist_dr = {
 "D1":{"R1":6,"R2":13,"R3":12,"R4":11,"R5":11,"R6":15,"R7":14,"R8":13,"R9":12,"R10":15,"R11":14},
 "D2":{"R1":10,"R2":10,"R3":10,"R4":12,"R5":10,"R6":12,"R7":12,"R8":11,"R9":13,"R10":11,"R11":13},
 "D3":{"R1":12,"R2":13,"R3":9,"R4":9,"R5":13,"R6":8,"R7":10,"R8":10,"R9":8,"R10":17,"R11":10},
 "D4":{"R1":13,"R2":11,"R3":12,"R4":15,"R5":14,"R6":13,"R7":14,"R8":10,"R9":13,"R10":16,"R11":11},
 "D5":{"R1":10,"R2":9,"R3":11,"R4":11,"R5":15,"R6":15,"R7":10,"R8":9,"R9":16,"R10":17,"R11":10},
 "D6":{"R1":6,"R2":10,"R3":14,"R4":17,"R5":9,"R6":13,"R7":9,"R8":13,"R9":18,"R10":18,"R11":10}
}



m = Model("Plant_DC_Retail")

x = m.addVars(plants, dcs, lb=0, name="x")       # Plant → DC
y = m.addVars(dcs, retails, lb=0, name="y")     # DC → Retail


m.setObjective(
    quicksum(dist_pd[p][d] * x[p,d] for p in plants for d in dcs) +
    quicksum((dist_dr[d][r] + dc_handle[d]) * y[d,r] for d in dcs for r in retails) +
    quicksum(dc_cost[d] * dc_open[d] for d in dcs),
    GRB.MINIMIZE
)


for p in plants:
    m.addConstr(quicksum(x[p,d] for d in dcs) <= plant_cap[p])

for r in retails:
    m.addConstr(quicksum(y[d,r] for d in dcs) == ret_demand[r])

for d in dcs:
    m.addConstr(quicksum(x[p,d] for p in plants) == quicksum(y[d,r] for r in retails))

for d in dcs:
    m.addConstr(quicksum(y[d,r] for r in retails) <= dc_cap[d] * dc_open[d])


m.optimize()

print("\nOptimal Cost:", m.objVal)


rows = []
for p in plants:
    for d in dcs:
        if x[p,d].x > 1e-6:
            rows.append([p, d, x[p,d].x])

for d in dcs:
    for r in retails:
        if y[d,r].x > 1e-6:
            rows.append([d, r, y[d,r].x])

df.head(20)


Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 26 rows, 84 columns and 234 nonzeros (Min)
Model fingerprint: 0x12b3b974
Model has 84 linear objective coefficients and an objective constant of 16500
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [8e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e+00, 7e+01]
Presolve removed 4 rows and 28 columns
Presolve time: 0.01s
Presolved: 22 rows, 56 columns, 156 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8268800e+04   2.162500e+01   0.000000e+00      0s
      18    1.9941100e+04   0.000000e+00   0.000000e+00      0s

Solved in 18 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.994110000e+04

Optimal Cost: 19941.1


,From,To,Flow
0,P1,D1,22.0
1,P1,D6,43.0
2,P2,D1,28.0
3,P3,D4,15.0
4,P3,D5,42.0
5,D1,R1,16.0
6,D1,R4,12.0
7,D1,R9,13.0
8,D1,R10,9.0
9,D4,R6,15.0
